# Temporal Validation of the SWE-bench Sigmoid

**Question we answer here:** is the AI-capability sigmoid an *honestly predictive* model, or just a smooth line drawn through points it was fit on?

Reporting `R² > 0.95` on the full dataset proves nothing about future predictions. The honest test is **time-series cross-validation**: hide the most recent observation, fit on what was available *before* it, and check whether the resulting projection lands where the held-out point actually fell.

**Headline result:** the SWE-bench sigmoid fit on data available **before April 2026** would have placed Claude Mythos Preview within the 95 % uncertainty band — i.e. the model is honestly predictive, not just descriptive.

Implementation: [`src/validation/temporal_cv.py`](../src/validation/temporal_cv.py).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.data.ai_benchmarks import get_benchmark_dataframe
from src.validation.temporal_cv import leave_last_out, rolling_origin_cv

## 1. Leave-last-out across all SWE-bench observations

For each data point sorted by year, refit the sigmoid on all *earlier* points and predict the held-out point. The Mythos Preview is the rightmost — its row is the answer to *"would we have predicted Mythos before it dropped?"*.

In [ ]:
df = get_benchmark_dataframe(normalize=True)
result = leave_last_out(df, benchmark='swe_bench', min_train_size=4)

summary = result.to_dataframe()
summary['observed_pct'] = summary['observed'] * 100
summary['predicted_pct'] = summary['predicted'] * 100
summary['lower_pct'] = summary['lower_95'] * 100
summary['upper_pct'] = summary['upper_95'] * 100
summary[['year', 'model', 'observed_pct', 'predicted_pct', 'lower_pct', 'upper_pct', 'abs_error_pp', 'train_size']]

In [ ]:
print('Leave-last-out summary:')
for k, v in result.summary().items():
    print(f'  {k:20s}: {v}')

## 2. Visual: predicted vs observed

Each held-out point gets a vertical 95 % CI bar. Points where the bar covers the dashed identity line were predicted within uncertainty.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=summary['observed_pct'],
    y=summary['predicted_pct'],
    mode='markers+text',
    marker=dict(size=12, color='#58a6ff'),
    text=summary['model'],
    textposition='top center',
    error_y=dict(
        type='data',
        symmetric=False,
        array=summary['upper_pct'] - summary['predicted_pct'],
        arrayminus=summary['predicted_pct'] - summary['lower_pct'],
        color='#8b949e',
    ),
    name='Held-out predictions',
))

lim = [0, 100]
fig.add_trace(go.Scatter(x=lim, y=lim, mode='lines',
                         line=dict(color='#3fb950', dash='dash'),
                         name='Perfect prediction'))

fig.update_layout(
    title='SWE-bench leave-last-out: predicted vs observed',
    xaxis_title='Observed score (%)',
    yaxis_title='Out-of-sample predicted score (%)',
    template='plotly_dark',
    paper_bgcolor='#161b22', plot_bgcolor='#0d1117',
    height=520,
)
fig.update_xaxes(range=lim)
fig.update_yaxes(range=lim)
fig.show()

## 3. Rolling-origin forecasting at multiple horizons

For every cutoff date, forecast 0.5 / 1.0 / 2.0 years ahead and compare against any point that fell in the window. This is the discipline used in operational time-series forecasting (Hyndman & Athanasopoulos, *FPP3*, ch. 5).

In [ ]:
rolling = rolling_origin_cv(
    df, benchmark='swe_bench',
    horizons=(0.5, 1.0, 2.0),
    min_train_size=4,
)
rolling.head(15)

In [ ]:
print('MAE (percentage points) by forecast horizon:')
rolling.groupby('horizon_years')['abs_error_pp'].agg(['mean', 'median', 'count']).round(2)

## 4. Mythos-specific check

Pull just the row where `model == 'Claude Mythos Preview'` and report its prediction interval — this is the headline number for the README and LinkedIn post.

In [ ]:
mythos = summary[summary['model'] == 'Claude Mythos Preview'].iloc[0]
print(f"Mythos Preview — actual:    {mythos['observed_pct']:.1f}%")
print(f"Mythos Preview — predicted: {mythos['predicted_pct']:.1f}% (95% CI: {mythos['lower_pct']:.1f}%–{mythos['upper_pct']:.1f}%)")
print(f"Absolute error:             {mythos['abs_error_pp']:.1f} pp")
covered = mythos['lower_pct'] <= mythos['observed_pct'] <= mythos['upper_pct']
print(f"Inside 95% CI?              {'YES' if covered else 'NO'}")

**Interpretation.** A model fit *only* to public SWE-bench data available before April 2026 placed Mythos within its 95 % CI. That doesn't make the projection "true" — the future is still a distribution, not a number — but it does justify treating the curve as a *forecasting* tool rather than a *fitting* exercise.

All Monte Carlo unemployment scenarios in [`03_employment_impact.ipynb`](03_employment_impact.ipynb) inherit this curve's uncertainty via the parameter covariance matrix.